# MLOps Training 2026/2027 — Task 1: Get the Data Into a Database

**Goal:** Download the Olist e-commerce dataset, load it into a PostgreSQL database, and verify it with queries and joins.

**Dataset:** [Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — a real, multi-table e-commerce dataset from a Brazilian marketplace. The tables are linked through `order_id`, `customer_id`, `product_id`, and `seller_id`.

**Business problem we're preparing for:** predict whether an order will be **delivered late or on time**, based on order, customer, product, seller, payment, and review information. This notebook does *not* solve that problem yet — it only builds the data foundation (database + verification) that later modeling tasks will build on.

**Structure of this notebook (matches the task steps):**
1. **Setup** — imports and locating the raw CSV files
2. **Step 2 — Download & Ingest** — connect to PostgreSQL and load the CSVs into tables
3. **Step 3 — Test it** — inspect the schema, run queries, join tables, and sanity-check the data against the business question


## 1. Setup

### Import core libraries
We import `pandas` (to read the CSVs and later run SQL queries into DataFrames) and `os` (to work with file paths). Printing the pandas version confirms the environment is ready before we touch any data.


In [11]:
import pandas as pd
import os

print("Pandas version:", pd.__version__)


Pandas version: 2.3.3


### Point to the local dataset folder
This defines `archive_path`, the local folder where the Olist CSV files (downloaded from Kaggle as described in Step 1 of the task) were extracted. Every later cell that reads a CSV uses this path, so it's set once at the top.



In [12]:
archive_path = r"C:\Users\HP\Desktop\MLOps\Task1\archive_2"

print("Data path:", archive_path)

Data path: C:\Users\HP\Desktop\MLOps\Task1\archive_2


###  Discover the CSV files
Before loading anything, we list every `.csv` file found in `archive_path` and print its size. This is a quick sanity check that the Kaggle download/extraction worked and that we can see all the Olist tables (orders, customers, products, sellers, order items, payments, reviews, etc.) before ingesting them.


In [13]:
import os

csv_files = [
    f for f in os.listdir(archive_path)
    if f.endswith(".csv")
]

print(f"Found {len(csv_files)} CSV files:")
print("=" * 70)

for i, file in enumerate(csv_files, 1):
    file_path = os.path.join(archive_path, file)
    file_size = os.path.getsize(file_path) / (1024**2)

    print(
        f"{i}. {file:45} "
        f"({file_size:8.2f} MB)"
    )

print("=" * 70)

Found 9 CSV files:
1. olist_customers_dataset.csv                   (    8.62 MB)
2. olist_geolocation_dataset.csv                 (   58.44 MB)
3. olist_orders_dataset.csv                      (   16.84 MB)
4. olist_order_items_dataset.csv                 (   14.72 MB)
5. olist_order_payments_dataset.csv              (    5.51 MB)
6. olist_order_reviews_dataset.csv               (   13.78 MB)
7. olist_products_dataset.csv                    (    2.27 MB)
8. olist_sellers_dataset.csv                     (    0.17 MB)
9. product_category_name_translation.csv         (    0.00 MB)


## 2. Step 2 — Download & Ingest: connect to PostgreSQL

### Configure the database connection
This sets the PostgreSQL connection parameters (host, port, database name, user, password) and builds a SQLAlchemy `engine`. The engine is the object every later cell uses to talk to the database — for both writing data (`to_sql`) and reading it back (`pd.read_sql_query`).



In [14]:
from sqlalchemy import create_engine

# PostgreSQL configuration
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist"
DB_USER = "postgres"
DB_PASSWORD = "123456"

# Create PostgreSQL connection
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)



### Test the database connection
Before loading any data, we confirm PostgreSQL is actually reachable by running `SELECT version();` and printing the result. This is the "Docker + database foundations" check from the task notes — if this cell fails, the Postgres container/service isn't running or the connection settings are wrong, and there's no point continuing.


In [15]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version();"))
        version = result.fetchone()[0]

        print("Successfully connected to PostgreSQL!")
        print("\nPostgreSQL version:")
        print(version)

except Exception as e:
    print("❌ Connection failed!")
    print(e)

Successfully connected to PostgreSQL!

PostgreSQL version:
PostgreSQL 17.10 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit


### Load every CSV into PostgreSQL as a table
This is the core ingestion step. For each CSV file found earlier, it:
1. Derives a clean table name from the filename (e.g. `olist_orders_dataset.csv` → `olist_orders`)
2. Reads the CSV into a pandas DataFrame
3. Writes the DataFrame into PostgreSQL with `df.to_sql(..., if_exists="replace")`, creating one table per CSV

After this cell runs, the raw Kaggle data physically lives inside the PostgreSQL database — completing the "database is running locally with the data inside" requirement from the task.


In [17]:
import pandas as pd
import os

# Get all CSV files
csv_files = [
    f for f in os.listdir(archive_path)
    if f.endswith(".csv")
]

print("Loading CSV files into PostgreSQL...")
print("=" * 80)

table_info = {}

for csv_file in csv_files:

    # Generate table name
    if "_dataset.csv" in csv_file:
        table_name = csv_file.replace("_dataset.csv", "")
    else:
        table_name = csv_file.replace(".csv", "")

    # Full CSV path
    file_path = os.path.join(
        archive_path,
        csv_file
    )

    # Read CSV
    df = pd.read_csv(file_path)

    # Store information
    table_info[table_name] = len(df)

    # Load DataFrame into PostgreSQL
    with engine.begin() as connection:
        connection.execute(
            text(f'DROP TABLE IF EXISTS "{table_name}" CASCADE')
        )

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )

    print(
        f"Loaded: {csv_file:45} → "
        f"table: {table_name:35} "
        f"({len(df):,} rows)"
    )

print("=" * 80)
print("All files loaded successfully into PostgreSQL!")

Loading CSV files into PostgreSQL...
Loaded: olist_customers_dataset.csv                   → table: olist_customers                     (99,441 rows)
Loaded: olist_geolocation_dataset.csv                 → table: olist_geolocation                   (1,000,163 rows)
Loaded: olist_orders_dataset.csv                      → table: olist_orders                        (99,441 rows)
Loaded: olist_order_items_dataset.csv                 → table: olist_order_items                   (112,650 rows)
Loaded: olist_order_payments_dataset.csv              → table: olist_order_payments                (103,886 rows)
Loaded: olist_order_reviews_dataset.csv               → table: olist_order_reviews                 (99,224 rows)
Loaded: olist_products_dataset.csv                    → table: olist_products                      (32,951 rows)
Loaded: olist_sellers_dataset.csv                     → table: olist_sellers                       (3,095 rows)
Loaded: product_category_name_translation.csv         →

## 3. Step 3 — Test it: inspect the schema and verify the data

### Confirm tables exist and count rows
Using SQLAlchemy's `inspect`, we list every table now present in the database and run a `COUNT(*)` on each one. This confirms all CSVs were loaded successfully and gives a first feel for the scale of the data (how many orders, customers, products, etc.).


In [18]:
from sqlalchemy import inspect, text

inspector = inspect(engine)

tables = inspector.get_table_names()

print("Tables in PostgreSQL:")
print("=" * 70)

total_rows = 0

for table_name in tables:

    with engine.connect() as connection:
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        )

        count = result.fetchone()[0]

    total_rows += count

    print(
        f"{table_name:<45} "
        f"{count:>12,} rows"
    )

print("=" * 70)

print(
    f"Database contains {len(tables)} tables "
    f"and {total_rows:,} total rows."
)

Tables in PostgreSQL:
olist_customers                                     99,441 rows
olist_geolocation                                1,000,163 rows
olist_orders                                        99,441 rows
olist_order_items                                  112,650 rows
olist_order_payments                               103,886 rows
olist_order_reviews                                 99,224 rows
olist_products                                      32,951 rows
olist_sellers                                        3,095 rows
product_category_name_translation                       71 rows
Database contains 9 tables and 1,550,922 total rows.


### Print the full database schema
For every table, we list its columns and their SQL types (alongside the row count again). This is how we build an understanding of **what each table contains and how the tables relate to each other** — the "understand the tables and how they relate" requirement from Step 1. It's the schema we'll rely on for every join and query below (e.g. spotting that `olist_orders`, `olist_order_items`, `olist_customers`, `olist_products`, `olist_sellers`, and `olist_order_payments` all connect via shared ID columns).


In [20]:
from sqlalchemy import inspect

inspector = inspect(engine)

print("POSTGRESQL DATABASE SCHEMA")
print("=" * 80)

tables = inspector.get_table_names()

for table_name in tables:

    columns = inspector.get_columns(table_name)

    with engine.connect() as connection:
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        )

        row_count = result.fetchone()[0]

    print(f"\n{table_name}")
    print(
        f"   Rows: {row_count:,} | "
        f"Columns: {len(columns)}"
    )

    print("   " + "-" * 70)

    for i, column in enumerate(columns, 1):

        print(
            f"   {i}. "
            f"{column['name']:<40} "
            f"{str(column['type']):<20}"
        )

POSTGRESQL DATABASE SCHEMA

olist_customers
   Rows: 99,441 | Columns: 5
   ----------------------------------------------------------------------
   1. customer_id                              TEXT                
   2. customer_unique_id                       TEXT                
   3. customer_zip_code_prefix                 BIGINT              
   4. customer_city                            TEXT                
   5. customer_state                           TEXT                

olist_geolocation
   Rows: 1,000,163 | Columns: 5
   ----------------------------------------------------------------------
   1. geolocation_zip_code_prefix              BIGINT              
   2. geolocation_lat                          DOUBLE PRECISION    
   3. geolocation_lng                          DOUBLE PRECISION    
   4. geolocation_city                         TEXT                
   5. geolocation_state                        TEXT                

olist_orders
   Rows: 99,441 | Columns: 8
   --

### 3.1 Join tests — one join per key relationship

The next four cells each test a single, specific join between two related tables. Doing them one relationship at a time (rather than one big query) makes it easy to confirm each foreign-key link works correctly before combining everything.

#### Join `orders` ↔ `customers`
Joins `olist_orders` to `olist_customers` on `customer_id`, pulling in order status/date plus the customer's city and state. This is the most basic relationship in the dataset: every order belongs to exactly one customer.


In [21]:
query = """
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    c.customer_id,
    c.customer_city,
    c.customer_state
FROM olist_orders o
JOIN olist_customers c 
    ON o.customer_id = c.customer_id
LIMIT 5;
"""

df = pd.read_sql_query(
    query,
    engine
)

print(df.to_string(index=False))

                        order_id order_status order_purchase_timestamp                      customer_id   customer_city customer_state
2711a938db643b3f0b62ee2c8a2784aa    delivered      2017-12-22 00:17:37 29cb486c739f9774c8eb542e07b56cd2        brasilia             DF
3bc77ce8be27211bac313c2daa402d1a    delivered      2017-04-06 22:39:29 bf141bf67fbe428d558bcf0e018eab60  belo horizonte             MG
10c320f977c6a18f91b2d14be13128c6    delivered      2017-05-09 20:48:59 b673f0597cb0c4d12778f731045f361a        gravatai             RS
0a4a2fccb27bd83a892fa503987a595b    delivered      2017-04-20 20:42:44 6772a0a230a2667d16c3620f000e1348     santa luzia             PB
e4de6d53ecff736bc68804b0b6e9f635    delivered      2017-10-16 14:56:50 9f6618c17568ac301465fe7ad056c674 antonio cardoso             BA


####  Join `orders` ↔ `order_items` ↔ `products`
A three-table join: an order can have multiple items (`olist_order_items`), and each item references a product (`olist_products`). This verifies the order→item→product chain and surfaces product category, price, and freight value per item.


In [22]:
query = """
SELECT 
    o.order_id,
    o.order_purchase_timestamp,
    oi.product_id,
    p.product_category_name,
    oi.price,
    oi.freight_value
FROM olist_orders o
JOIN olist_order_items oi 
    ON o.order_id = oi.order_id
JOIN olist_products p 
    ON oi.product_id = p.product_id
LIMIT 5;
"""

df = pd.read_sql_query(query, engine)

print(df.to_string(index=False))

                        order_id order_purchase_timestamp                       product_id product_category_name  price  freight_value
00010242fe8c5a6d1ba2dd792cb16214      2017-09-13 08:59:02 4244733e06e7ecb4970a6e2683c13e61            cool_stuff  58.90          13.29
00018f77f2f0320c557190d7a144bdd3      2017-04-26 10:53:06 e5f2d52b802189ee658865ca93d83a8f              pet_shop 239.90          19.93
000229ec398224ef6ca0657da4fc703e      2018-01-14 14:33:31 c777355d18b72b67abbeef9df44fd0fd      moveis_decoracao 199.00          17.87
00024acbcdf0a6daa1e931b038114c75      2018-08-08 10:00:35 7634da152a4610f1595efa32f14722fc            perfumaria  12.99          12.79
00042b26cf59d7ce69dfabb4e55b4fd9      2017-02-04 13:57:51 ac6c3623068f30de03045865e4e10089    ferramentas_jardim 199.90          18.14


#### Join `order_items` ↔ `sellers`
Joins order items to `olist_sellers` on `seller_id`, showing which seller (and their city/state) fulfilled each item. This confirms the seller-side relationship, which matters for the delivery-time problem since seller location affects shipping distance.


In [23]:
query = """
SELECT 
    oi.order_id,
    oi.order_item_id,
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    oi.price
FROM olist_order_items oi
JOIN olist_sellers s 
    ON oi.seller_id = s.seller_id
LIMIT 5;
"""

df = pd.read_sql_query(query, engine)

print(df.to_string(index=False))

                        order_id  order_item_id                        seller_id   seller_city seller_state  price
00010242fe8c5a6d1ba2dd792cb16214              1 48436dade18ac8b2bce089ec2a041202 volta redonda           SP  58.90
00018f77f2f0320c557190d7a144bdd3              1 dd7ddc04e1b6c2c614352b383efe2d36     sao paulo           SP 239.90
000229ec398224ef6ca0657da4fc703e              1 5b51032eddd242adc84c38acab88f23d borda da mata           MG 199.00
00024acbcdf0a6daa1e931b038114c75              1 9d7a1d34a5052409006425275ba1c2b4        franca           SP  12.99
00042b26cf59d7ce69dfabb4e55b4fd9              1 df560393f3a51e74553ab94004ba5c87        loanda           PR 199.90


#### Join `orders` ↔ `order_payments`
Joins orders to `olist_order_payments` on `order_id`. An order can have multiple payment records (e.g. split payments), so this checks that payment method, value, and installment data can be linked back to a specific order.


In [24]:
query = """
SELECT 
    o.order_id,
    o.order_purchase_timestamp,
    op.payment_sequential,
    op.payment_type,
    op.payment_value,
    op.payment_installments
FROM olist_orders o
JOIN olist_order_payments op 
    ON o.order_id = op.order_id
LIMIT 5;
"""

df = pd.read_sql_query(query, engine)

print(df.to_string(index=False))

                        order_id order_purchase_timestamp  payment_sequential payment_type  payment_value  payment_installments
b81ef226f3fe1789b1e8b2acac839d17      2018-04-25 22:01:49                   1  credit_card          99.33                     8
a9810da82917af2d9aefd1278f1dcfa0      2018-06-26 11:01:38                   1  credit_card          24.39                     1
25e8ea4e93396b6fa0d3dd708e76c1bd      2017-12-12 11:19:55                   1  credit_card          65.71                     1
ba78997921bbcdc1373bb41e913ab953      2017-12-06 12:04:06                   1  credit_card         107.78                     8
42fdf880ba16b47b59251dd489d4441a      2018-05-21 13:59:17                   1  credit_card         128.45                     2


### 3.2 Business/exploratory queries — connecting the data to the actual problem

These next queries go a step further than plain joins: they start answering questions directly tied to the task's business problem (**predicting late vs. on-time delivery**). This is optional "explore if you want" territory per the task notes, but it's useful here to prove we *understand* the problem, not just the schema.

####  Distribution of order statuses
Counts how many orders fall into each `order_status` (e.g. `delivered`, `shipped`, `canceled`) and their percentage share. This matters because only **delivered** orders have a real delivery date to compare against the estimate — so this defines which subset of the data the late/on-time label can even be computed on.


In [25]:
query = """
SELECT 
    order_status,
    COUNT(*) AS count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM olist_orders
GROUP BY order_status
ORDER BY count DESC;
"""

df = pd.read_sql_query(query, engine)

print(df.to_string(index=False))

order_status  count  percentage
   delivered  96478       97.02
     shipped   1107        1.11
    canceled    625        0.63
 unavailable    609        0.61
    invoiced    314        0.32
  processing    301        0.30
     created      5        0.01
    approved      2        0.00


####  Overall late vs. on-time delivery rate
For all `delivered` orders, compares `order_delivered_customer_date` to `order_estimated_delivery_date` to compute how many were late vs. on time, and the overall late percentage. This is essentially a first, single-number look at the target variable for the future prediction task.


In [26]:
query = """
SELECT 
    COUNT(*) AS total_delivered_orders,

    SUM(
        CASE 
            WHEN order_delivered_customer_date 
                 > order_estimated_delivery_date
            THEN 1 
            ELSE 0 
        END
    ) AS late_deliveries,

    SUM(
        CASE 
            WHEN order_delivered_customer_date 
                 <= order_estimated_delivery_date
            THEN 1 
            ELSE 0 
        END
    ) AS on_time_deliveries,

    ROUND(
        100.0 * SUM(
            CASE 
                WHEN order_delivered_customer_date 
                     > order_estimated_delivery_date
                THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_percentage

FROM olist_orders

WHERE order_status = 'delivered';
"""

result = pd.read_sql_query(query, engine)

print(result.to_string(index=False))

 total_delivered_orders  late_deliveries  on_time_deliveries  late_percentage
                  96478             7826               88644             8.11


####  Late delivery rate by customer state
Breaks the same late/on-time logic down by `customer_state`, ranking states by late-delivery percentage. This hints at geography as a likely predictive feature (e.g. orders shipped to distant states may run late more often), which is useful context for the modeling task that comes later.


In [27]:
query = """
SELECT 
    c.customer_state,
    COUNT(*) AS total_orders,

    SUM(
        CASE 
            WHEN o.order_delivered_customer_date 
                 > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE 
                WHEN o.order_delivered_customer_date 
                     > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_percentage

FROM olist_orders o

JOIN olist_customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_state

ORDER BY late_percentage DESC

LIMIT 15;
"""

df = pd.read_sql_query(query, engine)

print(df.to_string(index=False))

customer_state  total_orders  late_orders  late_percentage
            AL           397           95            23.93
            MA           717          141            19.67
            PI           476           76            15.97
            CE          1279          196            15.32
            SE           335           51            15.22
            BA          3256          457            14.04
            RJ         12350         1664            13.47
            TO           274           35            12.77
            PA           946          117            12.37
            ES          1995          244            12.23
            RR            41            5            12.20
            MS           701           81            11.55
            PB           517           57            11.03
            PE          1593          172            10.80
            RN           474           51            10.76


#### Draft an "ML-ready" query
Builds a single wide query that joins `orders`, `customers`, `order_items`, `order_reviews`, and `order_payments` together, aggregates them to one row per order, and computes the `is_late` label alongside features like number of items, total price, freight, number of sellers, average review score, and total payment. This is a preview of the feature table a future modeling notebook would start from — it isn't the modeling step itself, just a proof that all the tables can be combined into one order-level dataset.



In [28]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    c.customer_city,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_estimated_delivery_date,
    o.order_delivered_customer_date,

    CASE 
        WHEN o.order_delivered_customer_date 
             > o.order_estimated_delivery_date
        THEN 1 
        ELSE 0 
    END AS is_late,

    COUNT(DISTINCT oi.product_id) AS num_items,

    SUM(oi.price) AS total_price,

    SUM(oi.freight_value) AS total_freight,

    COUNT(DISTINCT oi.seller_id) AS num_sellers,

    AVG(orv.review_score) AS avg_review_score,

    SUM(op.payment_value) AS total_payment

FROM olist_orders o

LEFT JOIN olist_customers c
    ON o.customer_id = c.customer_id

LEFT JOIN olist_order_items oi
    ON o.order_id = oi.order_id

LEFT JOIN olist_order_reviews orv
    ON o.order_id = orv.order_id

LEFT JOIN olist_order_payments op
    ON o.order_id = op.order_id

WHERE o.order_status = 'delivered'

GROUP BY 
    o.order_id,
    o.customer_id,
    c.customer_city,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_estimated_delivery_date,
    o.order_delivered_customer_date

LIMIT 10;
"""

df_ml = pd.read_sql_query(
    query,
    engine
)

print("ML-READY DATASET")
print("=" * 80)

print(df_ml.to_string(index=False))

print("\nShape:", df_ml.shape)

ML-READY DATASET
                        order_id                      customer_id         customer_city customer_state order_purchase_timestamp   order_approved_at order_estimated_delivery_date order_delivered_customer_date  is_late  num_items  total_price  total_freight  num_sellers  avg_review_score  total_payment
00010242fe8c5a6d1ba2dd792cb16214 3ce436f183e68e07877b285a838db11a campos dos goytacazes             RJ      2017-09-13 08:59:02 2017-09-13 09:45:35           2017-09-29 00:00:00           2017-09-20 23:43:48        0          1        58.90          13.29            1               5.0          72.19
00018f77f2f0320c557190d7a144bdd3 f6dd3ec061db4e3987629fe6b26e5cce       santa fe do sul             SP      2017-04-26 10:53:06 2017-04-26 11:05:13           2017-05-15 00:00:00           2017-05-12 16:04:24        0          1       239.90          19.93            1               4.0         259.83
000229ec398224ef6ca0657da4fc703e 6489ae5e4333f3693df5ad4372dab6d3         par

## 4. Final check — confirming the "Done when" criteria

### Task 1 completion check
This closing cell re-verifies, in one place, everything the task's "Done when" checklist asks for:
- **Database is running locally with the data inside** → lists all tables and their row counts
- **You can query the tables and join them** → re-runs the `orders` ↔ `customers` join and reports the row count
- **You understand the problem we're trying to solve** → recomputes the overall late vs. on-time delivery split for delivered orders

If this cell runs end-to-end without errors, Task 1 is complete.


In [29]:
from sqlalchemy import inspect, text

print("=" * 80)
print("POSTGRESQL TASK 1 COMPLETION CHECK")
print("=" * 80)

# Get tables
inspector = inspect(engine)
tables = inspector.get_table_names()

print(f"\nDatabase contains {len(tables)} tables")

# Row counts
print("\nTable row counts:")

for table_name in tables:

    with engine.connect() as connection:
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        )

        count = result.fetchone()[0]

    print(
        f"   • {table_name:<45} "
        f"{count:>12,} rows"
    )

# Test JOIN
query = """
SELECT COUNT(*) AS count
FROM olist_orders o
JOIN olist_customers c
    ON o.customer_id = c.customer_id;
"""

result = pd.read_sql_query(
    query,
    engine
)

print("\nOrders + Customers JOIN:")
print(
    f"   {result.iloc[0]['count']:,} rows"
)

# Prediction problem
query = """
SELECT 
    COUNT(*) AS total,

    SUM(
        CASE 
            WHEN order_delivered_customer_date 
                 > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late

FROM olist_orders

WHERE order_status = 'delivered';
"""

result = pd.read_sql_query(
    query,
    engine
).iloc[0]

total = result["total"]
late = result["late"]

late_pct = 100 * late / total

print("\n Prediction problem:")
print(f"   • Delivered orders: {total:,}")
print(f"   • Late orders: {late:,}")
print(f"   • Late percentage: {late_pct:.2f}%")
print(f"   • On-time percentage: {100-late_pct:.2f}%")

print("\n" + "=" * 80)
print("ALL COMPLETION CRITERIA MET!")
print("=" * 80)

POSTGRESQL TASK 1 COMPLETION CHECK

Database contains 9 tables

Table row counts:
   • olist_customers                                     99,441 rows
   • olist_geolocation                                1,000,163 rows
   • olist_orders                                        99,441 rows
   • olist_order_items                                  112,650 rows
   • olist_order_payments                               103,886 rows
   • olist_order_reviews                                 99,224 rows
   • olist_products                                      32,951 rows
   • olist_sellers                                        3,095 rows
   • product_category_name_translation                       71 rows

Orders + Customers JOIN:
   99,441 rows

 Prediction problem:
   • Delivered orders: 96,478
   • Late orders: 7,826
   • Late percentage: 8.11%
   • On-time percentage: 91.89%

ALL COMPLETION CRITERIA MET!


In [30]:
import sqlite3

sqlite_conn = sqlite3.connect("olist.db")

for table_name in tables:  # from your inspector.get_table_names()
    df = pd.read_sql_query(f'SELECT * FROM "{table_name}"', engine)
    df.to_sql(table_name, sqlite_conn, if_exists="replace", index=False)

sqlite_conn.close()
print(" olist.db created with all tables")

 olist.db created with all tables
